# 01 · Data check

Loads every dataset and checks NASA POWER weather against the Kaggle weather columns.

Status: Kaggle ✅ · NASA POWER ✅ · Agmarknet prices ✅ · DES district yield ✅ (replaces ICRISAT, which is offline)

In [1]:
import sys
sys.path.insert(0, '../project')
import pandas as pd
import data
pd.set_option('display.width', 200)

## 1. Kaggle Crop Recommendation

In [2]:
crops = data.kaggle_crops()
print(crops.shape)
print(crops.label.value_counts().unique(), 'rows per crop')
crops.describe().T[['min', 'mean', 'max']]

(2200, 8)
[100] rows per crop


,min,mean,max
N,0.000000,50.551818,140.000000
P,5.000000,53.362727,145.000000
K,5.000000,48.149091,205.000000
temperature,8.825675,25.616244,43.675493
humidity,14.258040,71.481779,99.981876
ph,3.504752,6.469480,9.935091
rainfall,20.211267,103.463655,298.560117


## 2. NASA POWER weather

Monthly values for 2015–2024. Rainfall is converted to mm per month.

In [3]:
data.nasa_weather(20.00, 73.79).tail()  # Nashik

,rainfall,humidity,temperature
2024-08-01,591.48,91.38,24.37
2024-09-01,425.40,91.63,23.67
2024-10-01,119.04,85.19,24.03
2024-11-01,0.00,74.37,19.98
2024-12-01,15.19,67.11,20.10


## 3. Does real weather look like Kaggle's weather?

Kaggle never says what period its temperature, humidity and rainfall cover. For each crop we pick a place known for growing it and compare Kaggle's average with NASA POWER:
- **year**: mean over all months (rainfall = mean monthly total)
- **season**: mean over the crop's growing months

In [4]:
kag = crops.groupby('label')[['temperature', 'humidity', 'rainfall']].mean()
places = {  # crop: (place, lat, lon, growing months)
    'rice': ('Raipur', 21.25, 81.63, [6, 7, 8, 9]),
    'cotton': ('Yavatmal', 20.39, 78.12, [6, 7, 8, 9]),
    'chickpea': ('Vidisha', 23.52, 77.81, [11, 12, 1, 2]),
    'apple': ('Shimla', 31.10, 77.17, [4, 5, 6, 7, 8, 9]),
    'coffee': ('Madikeri', 12.42, 75.74, list(range(1, 13))),
    'grapes': ('Nashik', 20.00, 73.79, list(range(1, 13))),
    'mango': ('Ratnagiri', 16.99, 73.30, list(range(1, 13))),
    'maize': ('Davangere', 14.47, 75.92, [6, 7, 8, 9]),
}
rows = []
for crop, (place, lat, lon, months) in places.items():
    w = data.nasa_weather(lat, lon)
    year, season = w.mean(), w[w.index.month.isin(months)].mean()
    for col in ['temperature', 'humidity', 'rainfall']:
        rows.append({'crop': crop, 'place': place, 'feature': col, 'kaggle': kag.loc[crop, col],
                     'nasa_year': year[col], 'nasa_season': season[col]})
cmp = pd.DataFrame(rows).round(1)
cmp['err_year'] = (cmp.nasa_year - cmp.kaggle).abs()
cmp['err_season'] = (cmp.nasa_season - cmp.kaggle).abs()
cmp.groupby('feature')[['err_year', 'err_season']].mean().round(1)

,err_year,err_season
feature,,
humidity,23.0,19.2
rainfall,46.2,78.8
temperature,2.7,2.5


In [5]:
cmp.pivot(index='crop', columns='feature', values=['kaggle', 'nasa_year'])

kaggle                      nasa_year                     
feature  humidity rainfall temperature  humidity rainfall temperature
crop                                                                 
apple        92.3    112.7        22.6      56.2    105.8        20.8
chickpea     16.9     80.1        18.9      55.2    110.3        25.1
coffee       58.9    158.1        25.5      78.7    200.8        24.2
cotton       79.8     80.4        24.0      56.1     86.6        26.9
grapes       81.9     69.6        23.8      62.7    158.4        24.5
maize        65.1     84.8        22.4      67.0     82.1        25.3
mango        50.2     94.7        31.2      74.7    166.2        27.7
rice         82.3    236.2        23.7      61.9    115.9        26.0

In [6]:
out = data.ROOT / 'research' / 'results' / '2026-10-29_weather_vs_kaggle.csv'
out.parent.mkdir(parents=True, exist_ok=True)
cmp.to_csv(out, index=False)
out

PosixPath('/home/ankushss/Desktop/Office/Namebia/Model-New/research/results/2026-10-29_weather_vs_kaggle.csv')

## 4. Mandi prices (Agmarknet, India Data Portal mirror)

Monthly median modal price per crop and state, Jan 2021 – May 2026. Needs the two APMC zips in `data/raw/apmc/` (manual download).

In [7]:
prices = data.mandi_prices()
print(prices.shape, prices.month.min().date(), prices.month.max().date())
prices.groupby('crop').agg(states=('state', 'nunique'), months=('month', 'nunique'), median_price=('price', 'median'))

(16969, 5) 2021-01-01 2026-05-01


,states,months,median_price
crop,,,
apple,25,65,8650.00
banana,27,65,2500.00
blackgram,20,65,6825.00
chickpea,19,65,5525.00
coconut,13,65,2292.50
coffee,2,65,15425.00
cotton,13,65,7200.00
grapes,23,65,6000.00
jute,7,65,5450.00


## 5. District yield (DES area-production-yield, replaces ICRISAT)

Mean of each district's last 5 reported years. `last_year` shows how stale it is: fruits mostly stop by 2015.

In [8]:
yields = data.district_yield()
yields.groupby('crop').agg(districts=('district_code', 'nunique'), yield_t_ha=('yield_t_ha', 'median'), newest=('last_year', 'max'))

,districts,yield_t_ha,newest
crop,,,
banana,409,38.800000,2022-2023
blackgram,669,0.653463,2022-2023
chickpea,643,1.035635,2022-2023
coconut,180,5.946834,2022-2023
coffee,3,0.730097,2002-2003
cotton,462,0.980873,2022-2023
grapes,70,22.944779,2014-2015
jute,185,1.440000,2022-2023
kidneybeans,9,1.253012,2013-2014


## 6. District coordinates

Median latitude/longitude of each district's APMC markets, for NASA POWER lookups.

In [9]:
coords = data.district_coords()
d = yields.drop_duplicates('district_code')
print(len(d), 'districts,', d.district_code.isin(coords.district_code).sum(), 'with coordinates')
d.merge(coords, on='district_code').query('district_name == "Nashik"')

739 districts, 625 with coordinates


,crop,district_code,yield_t_ha,area_ha,years,last_year,state_name,district_name,lat,lon,markets
557,blackgram,487,0.627241,5347.447,5,2022-2023,Maharashtra,Nashik,20.283757,74.205776,60
